# 01_bronze
Calls the Wikimedia APIs for the two French articles and appends each raw response, unchanged, to `nba_demand.bronze.api_responses`. Append-only: every run adds a new batch; silver reads the latest one.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nba_demand.bronze.api_responses (
  batch_id      STRING    COMMENT 'One id per pipeline run',
  ingested_at   TIMESTAMP COMMENT 'When the response was received (UTC)',
  source        STRING    COMMENT 'mediawiki_title_lookup or wikimedia_pageviews',
  lang          STRING    COMMENT 'Wikipedia language edition',
  topic         STRING    COMMENT 'basketball or nba',
  input_title   STRING    COMMENT 'Article title sent in the request',
  request_url   STRING    COMMENT 'Full URL called',
  http_status   INT       COMMENT 'HTTP status code',
  response_body STRING    COMMENT 'Raw response text, exactly as received'
)
COMMENT 'Bronze: raw Wikimedia API responses. Append-only, never cleaned.';

In [0]:
import os, sys, uuid
from datetime import date, datetime, timedelta, timezone
import requests

sys.path.append(os.path.abspath(".."))      # repo root of this Git folder
import src.collect_wikipedia as cw          # Project 1 collector: retry logic, title parsing, URL encoding

cw.HEADERS["User-Agent"] = "nba-europe-demand-monitoring/0.1 (github.com/razvan-m-30; student project)"

LANG = "fr"
ARTICLES = {"basketball": "Basket-ball", "nba": "National Basketball Association"}
batch_id = str(uuid.uuid4())
end_date = (date.today() - timedelta(days=1)).strftime("%Y%m%d")
session = requests.Session()
rows = []

def record(source, topic, title, resp):
    rows.append((batch_id, datetime.now(timezone.utc), source, LANG, topic, title,
                 resp.url, resp.status_code, resp.text))

for topic, title in ARTICLES.items():
    # 1. Title lookup: the pageviews API doesn't follow redirects, so use the canonical title
    resp = cw._request_with_retry(
        session, f"https://{LANG}.wikipedia.org/w/api.php",
        params={"action": "query", "titles": title, "redirects": 1, "format": "json"})
    record("mediawiki_title_lookup", topic, title, resp)
    res = cw.parse_title_resolution(LANG, "France", topic, title, resp.json())
    if res.missing:
        raise ValueError(f"Article not found on {LANG}.wikipedia: {title}")  # never substitute

    # 2. Daily pageviews, 2019-01-01 to yesterday
    url = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
           f"{LANG}.wikipedia/all-access/user/{cw.encode_title_for_path(res.canonical_title)}"
           f"/daily/{cw.PAGEVIEWS_START}/{end_date}")
    resp = cw._request_with_retry(session, url)
    record("wikimedia_pageviews", topic, res.canonical_title, resp)

schema = ("batch_id string, ingested_at timestamp, source string, lang string, topic string, "
          "input_title string, request_url string, http_status int, response_body string")
spark.createDataFrame(rows, schema).write.mode("append").saveAsTable("nba_demand.bronze.api_responses")
print(f"Batch {batch_id}: appended {len(rows)} responses")

In [0]:
%sql
SELECT ingested_at, source, topic, input_title, http_status,
       length(response_body) AS body_chars,
       json_array_length(get_json_object(response_body, '$.items')) AS days_returned
FROM nba_demand.bronze.api_responses
ORDER BY ingested_at DESC;